In [15]:
import torch.nn as nn
import torchvision

class SimpleDigitCNN(nn.Module):
    def __init__(self, num_slots: int = 5, num_classes: int = 11):
        super().__init__()
        self.backbone = torchvision.models.mobilenet_v3_small(weights="IMAGENET1K_V1").features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.5)
        self.head = nn.Linear(576, num_slots * num_classes)
        self.num_slots = num_slots
        self.num_classes = num_classes
    
    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        x = self.head(x)
        return x.view(-1, self.num_slots, self.num_classes)

In [16]:
import pandas as pd
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as F
import torch.nn.functional as nnF
import torch

class DigitDataset(Dataset):
    def __init__(self, csv_path: Path, target_height: int = 128, skip_empty: bool = False, transform=None) -> None:
        self.csv_dir = Path(csv_path).parent
        self.data = pd.read_csv(csv_path, header=None, names=["image", "label"], dtype={"label": str})
        if skip_empty:
            self.data = self.data[self.data["label"].notna() & (self.data["label"].str.strip() != "")]
        self.target_height = target_height
        self.transform = transform


    def __len__(self):
        return len(self.data)
    

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        image = Image.open(self.csv_dir / row["image"]).convert("RGB")

        w, h = image.size
        new_w = int(w * self.target_height / h)
        image = image.resize((new_w, self.target_height))

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = F.to_tensor(image)
            image = F.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        label = [int(c) for c in row["label"]] if pd.notna(row["label"]) and str(row["label"]) != "" else []

        return image, label, new_w


def pad_label(label, length=5, blank=10):
    return (label + [blank] * length)[:length]


def collate_fn(batch):
    images, labels, widths = zip(*batch)

    max_width = max(img.shape[2] for img in images)
    padded = [nnF.pad(img, (0, max_width - img.shape[2])) for img in images]

    return torch.stack(padded), list(labels), list(widths)

In [17]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path

criterion = nn.CrossEntropyLoss(ignore_index=10)

def train(model: SimpleDigitCNN, train_dataset: DigitDataset, val_dataset: DigitDataset, epochs: int = 50, lr: float = 1e-4, batch_size: int = 16):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels, widths in train_loader:
            images = images.to(device)
            padded = torch.tensor([pad_label(l) for l in labels], dtype=torch.long).to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits.view(-1, 11), padded.view(-1))

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, labels, widths in val_loader:
                images = images.to(device)
                padded = torch.tensor([pad_label(l) for l in labels], dtype=torch.long).to(device)
                logits = model(images)
                val_loss += criterion(logits.view(-1, 11), padded.view(-1)).item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}/{epochs} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")
    
    return model

In [18]:
import torchvision.transforms as transforms

model = SimpleDigitCNN()
data_dir = Path("data/digits")
train_dataset = DigitDataset(data_dir / "train.csv", skip_empty=True)
val_dataset = DigitDataset(data_dir / "val.csv", skip_empty=True)

dummy = torch.randn(1, 3, 128, 128)
with torch.no_grad():
    out = model(dummy)
print(out.shape)  # (batch, channels, H, W)

model = train(model, train_dataset, val_dataset)

torch.Size([1, 5, 11])
Epoch 1/50 | train loss: 2.3809 | val loss: 2.4610
Epoch 2/50 | train loss: 2.0801 | val loss: 2.4581
Epoch 3/50 | train loss: 1.8712 | val loss: 2.4892
Epoch 4/50 | train loss: 1.6782 | val loss: 2.5395
Epoch 5/50 | train loss: 1.4905 | val loss: 2.5761
Epoch 6/50 | train loss: 1.3323 | val loss: 2.5892
Epoch 7/50 | train loss: 1.2146 | val loss: 2.6152
Epoch 8/50 | train loss: 1.1080 | val loss: 2.6310
Epoch 9/50 | train loss: 1.0342 | val loss: 2.6216
Epoch 10/50 | train loss: 1.0057 | val loss: 2.5921
Epoch 11/50 | train loss: 0.9624 | val loss: 2.5916
Epoch 12/50 | train loss: 0.9243 | val loss: 2.5852
Epoch 13/50 | train loss: 0.8976 | val loss: 2.5836
Epoch 14/50 | train loss: 0.8398 | val loss: 2.5856
Epoch 15/50 | train loss: 0.8345 | val loss: 2.5875
Epoch 16/50 | train loss: 0.8147 | val loss: 2.5941
Epoch 17/50 | train loss: 0.8089 | val loss: 2.5905
Epoch 18/50 | train loss: 0.7989 | val loss: 2.5899
Epoch 19/50 | train loss: 0.7652 | val loss: 2.596

In [ ]:
def decode_simple(logits):
    # logits: (5, 11)
    probs = torch.softmax(logits, dim=1)
    indices = torch.argmax(probs, dim=1)  # (5,)
    digits = [i.item() for i in indices if i.item() != 10]
    return "".join(str(d) for d in digits)

model.eval()
for i in range(10):
    image, label, width = val_dataset[i]
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))  # (1, 5, 11)
        pred = decode_simple(logits[0])
    true = "".join(str(d) for d in label)
    print(f"pred: {pred} | true: {true}")

pred: 10416 | true: 114
pred: 13836 | true: 3128
pred: 64861 | true: 3128
pred: 96331 | true: 1138
pred: 94866 | true: 28
pred: 44866 | true: 1138
pred: 94861 | true: 1138
pred: 96866 | true: 1138
pred: 44816 | true: 1138
pred: 94361 | true: 3
